# VIIRS 야간 밝기 추출 — 읍면동 단위

전국 shapefile과 GEE에서 받은 VIIRS `.tif`를 이용해  
읍면동별 연간 평균 밝기를 계산하고 `viirs_emd_2019_2026.csv`로 저장

## 1. 환경 설정

In [16]:
import glob
import os
import zipfile

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import rasterize

SHP_DIR = '행정구역'
TIF_DIR = 'GEE'
EXTRACT_DIR = 'shp_extracted'
OUTPUT_CSV = 'viirs_emd_2019_2026.csv'

## 2. 전국 shapefile 합치기

`행정구역/` 폴더에 있는 시도별 zip 17개를 풀어서 하나의 GeoDataFrame으로 합치기

In [17]:
zip_files = glob.glob(f'{SHP_DIR}/*.zip')
print(f'zip {len(zip_files)}개')

gdfs = []
for zip_path in sorted(zip_files):
    sido_name = os.path.basename(zip_path).replace('LSMD_ADM_SECT_UMD_', '').replace('.zip', '')
    extract_path = f'{EXTRACT_DIR}/{sido_name}'
    os.makedirs(extract_path, exist_ok=True)

    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_path)

    shp_file = glob.glob(f'{extract_path}/*.shp')[0]
    gdf = gpd.read_file(shp_file, encoding='euc-kr')
    gdfs.append(gdf)
    print(f'  {sido_name}: {len(gdf)}개')

korea_gdf = pd.concat(gdfs, ignore_index=True)
korea_gdf.crs = gdfs[0].crs

print(f'\n전국 읍면동: {len(korea_gdf)}개, 좌표계: {korea_gdf.crs}')

zip 17개
  강원특별자치도: 601개
  경기: 747개
  경남: 546개
  경북: 525개
  광주: 202개
  대구: 212개
  대전: 177개
  부산: 192개
  서울: 467개
  세종: 33개
  울산: 84개
  인천: 156개
  전남: 421개
  전북특별자치도: 410개
  제주: 74개
  충남: 285개
  충북: 238개

전국 읍면동: 5370개, 좌표계: EPSG:5186


## 3. 좌표계 변환

shapefile은 EPSG:5186, `.tif`는 EPSG:4326이라 맞추기  
`zone_id` — 읍면동 구분 번호

In [18]:
korea_gdf = korea_gdf.to_crs('EPSG:4326')
korea_gdf['zone_id'] = korea_gdf.index + 1

print(f'좌표계: {korea_gdf.crs}')
korea_gdf[['EMD_CD', 'EMD_NM', 'zone_id']].head()

좌표계: EPSG:4326


,EMD_CD,EMD_NM,zone_id
0,51110115,소양로3가,1
1,51110122,교동,2
2,51110108,조양동,3
3,51110400,남산면,4
4,51110110,운교동,5


## 4. tif에서 읍면동별 밝기 추출

연도별 `.tif`마다 읍면동 폴리곤을 픽셀에 매핑한 뒤, 해당 픽셀들의 평균을 읍면동 밝기값으로

In [19]:
tif_files = sorted(glob.glob(f'{TIF_DIR}/*.tif'))
print(f'tif {len(tif_files)}개')

results = []

for tif_path in tif_files:
    year = os.path.basename(tif_path).replace('viirs_korea_', '').replace('.tif', '')
    print(f'{year}년...', end=' ')

    with rasterio.open(tif_path) as src:
        data = src.read(1).astype(float)
        data[data < 0] = np.nan

        zone_raster = rasterize(
            [(geom, zid) for geom, zid in zip(korea_gdf.geometry, korea_gdf.zone_id)],
            out_shape=src.shape,
            transform=src.transform,
            fill=0,
            dtype='int32',
        )

        for _, row in korea_gdf.iterrows():
            pixels = data[zone_raster == row['zone_id']]
            pixels = pixels[~np.isnan(pixels)]
            mean_val = float(np.mean(pixels)) if len(pixels) > 0 else None

            results.append({
                'EMD_CD': row['EMD_CD'],
                'EMD_NM': row['EMD_NM'],
                'year': int(year),
                'nightlight': round(mean_val, 4) if mean_val else None,
            })

    print('완료')

print(f'\n총 {len(results)}행')

tif 8개
2019년... 완료
2020년... 완료
2021년... 완료
2022년... 완료
2023년... 완료
2024년... 완료
2025년... 완료
2026년... 완료

총 42960행


## 5. CSV 저장

연도를 컬럼으로 펼쳐서 저장

In [20]:
df = pd.DataFrame(results)

df_pivot = df.pivot_table(
    index=['EMD_CD', 'EMD_NM'],
    columns='year',
    values='nightlight',
).reset_index()

years = sorted(df['year'].unique())
df_pivot.columns = ['EMD_CD', 'EMD_NM'] + [f'nightlight_{y}' for y in years]

print(df_pivot.shape)
df_pivot.head()

df_pivot.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f'\n{OUTPUT_CSV} 저장 완료')

(4516, 10)

viirs_emd_2019_2026.csv 저장 완료
